# Generating Figure 1 from Kaiser et al. 2020

This is the same message at the beginning of all jupyter notebooks in this directory. 

If you don't have the below packages, you obviously need to install them for this to work. If it doesn't work still it's extremely likely you have an outdated version of one of the packages. Alternatively, some of the histogram functions actually rely on not being the most recent version because they changed from "normed" to something else from my recollection. Or perhaps it was the other way. I am aware this was poor decision-making, but it works (if you use the right version). ¯\\_(ツ)_/¯

Also pretty much all of these commands are copied and pasted from another Jupyter notebook I made but contained tons of tries at doing this stuff (and unrelated efforts) so that's why a lot of the variables seem unnecessary to use.

In [1]:
from __future__ import print_function


import matplotlib
matplotlib.use('pdf')


import numpy as np
import matplotlib.pyplot as plt
import sys
import os
from astropy.io import fits
from glob import glob
from astropy.time import Time
from astropy import coordinates as coords
from astropy import units as u
from astropy import constants as const
from astropy import convolution as conv
from astropy.table import Table, Column
import scipy.interpolate as scinterp
import time
start = time.time()

import spec_plot_tools as spt
import cal_params as cp
import plot_spec as ps
import fix_strings as fs

print(os.getcwd())

filename: -f
sdss_names: ['/Users/BenKaiser/Desktop/SDSS_speclib/K7_+0.0_Dwarf.fits', '/Users/BenKaiser/Desktop/SDSS_speclib/K7_+0.5_Dwarf.fits', '/Users/BenKaiser/Desktop/SDSS_speclib/K7_+1.0_Dwarf.fits', '/Users/BenKaiser/Desktop/SDSS_speclib/K7_-0.5_Dwarf.fits', '/Users/BenKaiser/Desktop/SDSS_speclib/K7_-1.0_Dwarf.fits', '/Users/BenKaiser/Desktop/SDSS_speclib/M0_+0.0_Dwarf.fits', '/Users/BenKaiser/Desktop/SDSS_speclib/M0_+0.5_Dwarf.fits', '/Users/BenKaiser/Desktop/SDSS_speclib/M0_+1.0_Dwarf.fits', '/Users/BenKaiser/Desktop/SDSS_speclib/M0_-0.5_Dwarf.fits', '/Users/BenKaiser/Desktop/SDSS_speclib/M1_+0.0_Dwarf.fits', '/Users/BenKaiser/Desktop/SDSS_speclib/M1_+0.5_Dwarf.fits', '/Users/BenKaiser/Desktop/SDSS_speclib/M1_+1.0_Dwarf.fits', '/Users/BenKaiser/Desktop/SDSS_speclib/M1_-0.5_Dwarf.fits', '/Users/BenKaiser/Desktop/SDSS_speclib/M1_-1.0_Dwarf.fits', '/Users/BenKaiser/Desktop/SDSS_speclib/M2_+0.0_Dwarf.fits', '/Users/BenKaiser/Desktop/SDSS_speclib/M2_+0.5_Dwarf.fits', '/Users/BenKai

In [2]:
#target_dir= '/Users/BenKaiser/Desktop/GaiaJ1644m0449_paper/'
target_dir='/Users/BenKaiser/Desktop/PhD_dissertation/Thesis_Proposal/thesis_proposal_presentation/spectra_for_plots/'
target_spec_file='ravg_fwctb.GaiaJ0356m2255_20191101_poly2_400m1.fits'
target_spec_file2='ravg_fwctb.GaiaJ0356m2255_20191101_ctrlpoly2_400m1.fits'
#model_spec_file='J1644_fit_flambda.dms'
#target_spec_file= glob(target_dir+target_spec_file)
#model_spec_file= glob(target_dir+model_spec_file)
print(target_spec_file)
#print(model_spec_file)

ravg_fwctb.GaiaJ0356m2255_20191101_poly2_400m1.fits


In [3]:
#figure_output_dir='/Users/BenKaiser/Desktop/GaiaJ1644m0449_paper/First_Revision/figures'
#figure_output_dir='/Users/BenKaiser/Desktop/GaiaJ1644m0449_paper/Science_versions/Third_Revision/figures'
figure_output_dir='/Users/BenKaiser/Desktop/'

In [4]:
#figure_output_dir='/Users/BenKaiser/Desktop/GaiaJ1644m0449_paper/ApJ_reformat/figures'

I guess I have to just change the working directory instead of using a long file path for whatever reason...

In [5]:
os.chdir(target_dir)

In [6]:
target_spec, header, target_noise= spt.retrieve_spec(target_spec_file)
target_spec2,header2, target_noise2=spt.retrieve_spec(target_spec_file2)

In [7]:
print(target_noise.shape)

(2, 1675)


### *New addition 2021-06-04 for other stitching as a test (really should make a stitching script separate from this notebook... but I'm lazy...)

target_spec_file='ravg_fwctb.WD2356m209_400m1.fits'
target_spec_file2='ravg_fwctb.WD2356m209_20190601_ted_tellcorr_400m2.fits'
target_spec, header, target_noise= spt.retrieve_spec(target_spec_file)
target_spec2,header2, target_noise2=spt.retrieve_spec(target_spec_file2)

In [8]:
sm_target_spec=ps.convolve_spectrum(target_spec, header,kernel_type='gaussian',pix_width=0.5*header['see_sig'])
sm_target_spec2=ps.convolve_spectrum(target_spec2, header2,kernel_type='gaussian',pix_width=0.5*header['see_sig'])
#sm_target_spec=target_spec
#sm_target_spec2=target_spec2

In [9]:
trim_spot=6800
tsm_target_spec=sm_target_spec
tsm_target_spec2=sm_target_spec2
#tsm_target_spec=spt.clean_spectrum(target_spec,np.min(target_spec[0]), trim_spot,[])
#tsm_target_spec2=spt.clean_spectrum(target_spec2,trim_spot, np.max(target_spec2[0]),[])


In [16]:
plt.rc('font',size=18)
#fig= plt.figure(figsize=(15,10)) #from v1 description image size
spt.initiate_science_plot()
fig=plt.figure(figsize=(7.25,7.25*2./3),constrained_layout=False)
#spt.start_ApJ_fig(width_cols=2,constrained_layout=True, width_height=[7.25,7.25*2./3])
label_pos=0.8
label_pos2= 0.6
label_off=110

plt.axhline(y=0,linestyle=':',color='k')
plt.plot(tsm_target_spec[0],tsm_target_spec[1], label=fs.fix_display_string("Gaia J0356-2255"), color='k')
plt.plot(tsm_target_spec2[0],tsm_target_spec2[1], color='gray',label='Control Extraction')
#plt.plot(tmodel_spec[0],tmodel_spec[1], label='Best Fit Model', color='r',alpha=0.7)


plt.xlim(np.nanmin(target_spec[0]), np.nanmax(target_spec[0]))
plt.ylim(-0.3, 1.0)

k_spot=np.mean([7664.899016,7698.96445153])


plt.annotate('Na I D',xy=(5895.9241497669427, 0.4),xytext=(5895.9241497669427-label_off, label_pos), arrowprops=dict(arrowstyle='-'))
plt.annotate('Li I?',xy=(6707.9080032878719, 0.6),xytext=(6707.9080032878719-55, label_pos), arrowprops=dict(arrowstyle='-'))
plt.annotate('Ca II\nH & K?',xy=(3968.4672118153667, 0.4),xytext=(3968.4672118153667-120, label_pos2), arrowprops=dict(arrowstyle='-'))
plt.annotate('Ca I?',xy=(4226.7295809531952, 0.4),xytext=(4226.7295809531952-70, label_pos2), arrowprops=dict(arrowstyle='-'))
#plt.annotate('MgH band &\nNa-He QM line',xy=(5190, 0.16),xytext=(5190-260, label_pos-0.05), arrowprops=dict(arrowstyle='-['))
plt.annotate('MgH band?',xy=(5190, 0.7),xytext=(5190-210, label_pos), arrowprops=dict(arrowstyle='-['))
#plt.annotate('K I',xy=(k_spot, 0.25),xytext=(k_spot-60, label_pos), arrowprops=dict(arrowstyle='-['))





#plt.xlim(6000,8000)
#plt.ylabel(r'$f_{\lambda} (10^{-16} erg/ cm^{2}/s/ \AA)$')
#plt.ylabel(r'$f_{\lambda}$ ($10^{-16}$ erg/cm$^{2}$/s/$\AA$)')
#plt.ylabel(r'Flux ($10^{-16}$ erg/cm$^{2}$/s/$\mathrm{\AA}$)')

#plt.ylabel(r'$f_{\lambda} (10^{-16}$ erg cm$^{-2}$ s $^{-1}$ $\AA^{-1})$') #inverse power units instead of divisions
plt.ylabel(r'Flux (10$^{-16}$ erg cm$^{-2}$ s$^{-1}$ $\mathrm{\AA}^{-1}$)') #inverse power units instead of divisions




#plt.xlabel(r'$\lambda(\AA)$')
#plt.xlabel(r'Wavelength $(\AA)$')
plt.xlabel(r'Wavelength $(\mathrm{\AA})$')



#spt.show_plot(line_id='alkali', label_pos=0.25, convert_to_air=True)
spt.show_plot(line_id='',show_legend=False,actually_show=False)


print(os.getcwd())
os.chdir(figure_output_dir)
print(os.getcwd())
start = time.time()
print(start)
time_string=str(start).split('.')[0]

plt.legend(loc='best')
plt.savefig('fig_nightmare_control_'+time_string+'.pdf')#plt.grid(True)

plt.show()

/Users/BenKaiser/Desktop
/Users/BenKaiser/Desktop
1630206118.845834


/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/ipykernel_launcher.py:65: UserWarning: Matplotlib is currently using pdf, which is a non-GUI backend, so cannot show the figure.


os.chdir(target_dir)
trim_spot=6800
hdu1= fits.open(target_spec_file)
hdu2=fits.open(target_spec_file2)
waves1=hdu1[0].data
waves2=hdu2[0].data

#output_fits='stitched_J1644_spectrum.fits'
output_fits='stitched_J2356_spectrum.fits'

list_for_hdu=[]
for index in range(0,5):
    array1=np.copy(hdu1[index].data)
    array2=np.copy(hdu2[index].data)
    spec1=np.vstack([waves1,array1])
    spec2=np.vstack([waves2,array2])
    print(spec1.shape)
    tsm_spec1=spt.clean_spectrum(spec1,np.min(spec1[0]), trim_spot,[])
    tsm_spec2=spt.clean_spectrum(spec2,trim_spot, np.max(spec2[0]),[])
    stitched_spec=np.hstack([tsm_spec1,tsm_spec2])
    print(stitched_spec.shape)
    if index==0:
        out_hdu=fits.PrimaryHDU(stitched_spec[1], header=header)
        list_for_hdu.append(out_hdu)
    else:
        new_hdu=fits.ImageHDU(stitched_spec[1])
        list_for_hdu.append(new_hdu)
        
print(list_for_hdu)
hdulist=fits.HDUList(list_for_hdu)
os.chdir(figure_output_dir)
hdulist.writeto(output_fits)